# Vehicle source walkthrough — synthetic example

This fixture is invented for learning and testing. It contains no real listings or VINs.
We have not connected a data provider or implemented a sales estimator.

Follow the source CSV -> observed table -> validation -> latest listing counts.
Prices are **asking prices in USD**, timestamps are UTC, and statuses stay as received.
These example statuses are not a verified taxonomy for either retailer.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / "vehicle" / "src").is_dir():
    ROOT = ROOT / "vehicle"
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src" / "vehicle_tracker").is_dir(), "Launch from researchOS, vehicle/, or vehicle/notebooks/."
sys.path.insert(0, str(ROOT / "src"))
import vehicle_tracker

source_file = ROOT / "tests/fixtures/synthetic_listings.csv"
observed = pd.read_csv(source_file)
print("SYNTHETIC EXAMPLE ONLY:", source_file)
display(observed)


The observation key includes **retailer + listing ID + capture time**. The same
vehicle identifier at different retailers remains separate evidence. Missing prices
stay missing. Duplicate observations must be investigated before counting listings.


In [ ]:
key = ["retailer", "listing_id", "observed_at_utc"]
duplicate_rows = observed[observed.duplicated(key, keep=False)]
missing_keys = observed[key].isna().any(axis=1)
display(observed.isna().sum().rename("missing_values").to_frame())
display(duplicate_rows)
assert not missing_keys.any(), "Missing observation key: inspect the source."
assert duplicate_rows.empty, "Duplicate observation keys: inspect before analysis."

snapshots = observed.copy()
snapshots["observed_at_utc"] = pd.to_datetime(snapshots["observed_at_utc"], utc=True, errors="raise")
snapshots["asking_price_usd"] = pd.to_numeric(snapshots["asking_price_usd"], errors="raise")
# Parse dates before the second check: equivalent timestamp spellings are one key.
assert not snapshots.duplicated(key).any(), "Duplicate normalized observation keys."
display(snapshots)


The next table counts observations at the latest capture in this tiny example.
It is neither a complete-market inventory estimate nor a sales count.
A pending status does not establish a completed sale, and an absent observation
does not establish a removal. Real captures need explicit completeness evidence.


In [ ]:
latest_capture = snapshots["observed_at_utc"].max()
latest = snapshots[snapshots["observed_at_utc"].eq(latest_capture)].copy()
listing_counts = latest.groupby(["retailer", "native_status"], dropna=False).size().rename("observed_listings").reset_index()
display(latest)
display(listing_counts)
print("Synthetic listing counts only. Sales estimation is not implemented.")


## Move from the synthetic example to retained evidence
The example above teaches the schema. This retained browser sample is a real observation,
not a complete population. Notebook 10 follows its source fields into inventory checks;
notebook 20 follows a VIN through retained queries and SQLite. Read the [audit map](../docs/audit_guide.md)
for the functions and write boundaries. Asking price, pending status and sales are different measures.

In [ ]:
import json
from vehicle_tracker.carvana import parse_capture
retained_path = ROOT / 'tests/fixtures/carvana_browser_sample_20260907.json'
retained = json.loads(retained_path.read_text(encoding='utf-8'))
display(pd.json_normalize(retained['records']).head())
normalized_retained = parse_capture(retained)
display(normalized_retained[['listing_id', 'vin', 'asking_price_usd', 'availability_native', 'source_url']])
print('Retained observation:', retained['captured_at_utc'], retained_path)
print('Sample only; no population coverage or sale is established.')